# Módulo 07 — Reconhecimento de Dígitos Manuscritos (`MNIST`): De Redes Densas (`MLP`) a Convolucionais (`CNN`)

> **Pré-requisito recomendado:** Caso ainda não tenha estudado como funcionam camadas ocultas, funções de ativação diferenciáveis e o algoritmo *Backpropagation*, revise antes o **[Módulo 06 — Perceptron Multicamadas (`PMC`/`MLP`) e `Backpropagation`](06_pmc_e_backpropagation.ipynb)**. Para consultar qualquer termo técnico a qualquer momento, acesse o **[Glossário Completo (`GLOSSARIO.md`)](../GLOSSARIO.md)**.

---

## 1. O "Olá, Mundo!" da Visão Computacional e sua Ponte com a Engenharia

Até o Módulo 06, trabalhamos com **dados tabulares**: cada coluna de entrada ($x_1, x_2, \dots, x_n$) representava uma grandeza física bem definida medida por um sensor individual (temperatura, pressão, pH, comprimento de pétala).

Mas como uma Rede Neural Artificial processa uma **imagem**?

O reconhecimento automático de dígitos escritos à mão no clássico conjunto de dados **[MNIST](https://www.mnist.org/)** (*Modified National Institute of Standards and Technology*, organizado por Yann LeCun, Corinna Cortes e Christopher Burges) é universalmente conhecido como o problema **"Olá, Mundo!"** do Aprendizado Profundo (*Deep Learning*) e da Visão Computacional. Na história da Engenharia, ele nasceu de uma demanda real: **ler automaticamente códigos postais (CEPs) em envelopes nos correios e valores preenchidos à mão em cheques bancários**.

Na **Engenharia Industrial moderna**, exatamente a mesma matemática que vamos construir passo a passo neste caderno é aplicada diariamente em:
- **Inspeção Óptica Automatizada (Manufatura e Metalurgia):** Câmeras industriais na linha de montagem detectando trincas em cordões de solda, porosidades em peças fundidas ou falhas superficiais em chapas laminadas;
- **Controle e Automação:** Leitura automática de números de série gravados em blocos de motor, placas veiculares ou mostradores analógicos em plantas antigas;
- **Engenharia Elétrica e Termografia:** Classificação automática de pontos quentes (*hotspots*) em imagens infravermelhas de subestações e painéis fotovoltaicos;
- **Engenharia de Materiais:** Classificação de microestruturas metalográficas (perlita, ferrita, martensita) em imagens de microscopia.

---

## 2. Definição Formal do Problema e Roteiro deste Caderno

### O Problema
- **Entrada ($\mathbf{X}$):** Uma pequena matriz bidimensional de **$28 \times 28$ pixels em escala de cinza** (análoga à leitura de um sensor óptico CCD/CMOS de 8 bits, onde cada pixel assume um valor inteiro entre `0` = fundo escuro e `255` = traço claro intenso).
- **Objetivo ($y$):** Classificar qual algarismo manuscrito está desenhado na imagem, atribuindo um **rótulo inteiro de `0` a `9`** (um problema de **classificação multiclasse com 10 categorias mutuamente exclusivas**).

### As Duas Arquiteturas que Vamos Construir e Comparar
1. **`MLP` (*Multilayer Perceptron* / Rede Neural Totalmente Conectada):**
   - **Como funciona:** A matriz $28 \times 28$ é "achatada" (*Flatten*) como uma fita contínua em um **vetor 1D de $784$ números** ($28 \times 28 = 784$ pixels). Esse vetor atravessa camadas ocultas densas até chegar a uma camada de saída com **10 neurônios** (um especialista para cada dígito de `0` a `9`).
   - **Limitação crítica:** Ao transformar uma grade 2D em uma linha 1D, a rede **perde a noção espacial** (a relação geométrica entre pixels vizinhos acima, abaixo e nas diagonais), tratando cada coordenada de forma rígida e isolada.
2. **`CNN` (*Convolutional Neural Network* / Rede Neural Convolucional):**
   - **Como funciona:** É a arquitetura padrão-ouro para imagens e sinais espaciais. Em vez de achatar a imagem na entrada, utiliza pequenos **filtros convolucionais** (matrizes $3 \times 3$, equivalentes a filtros espaciais de Processamento Digital de Imagens) que **"varrem"** toda a superfície $28 \times 28$ para capturar bordas horizontais, hastes verticais, cantos, arcos e laços, **preservando integralmente a estrutura espacial 2D**.

### O Fluxo Completo de Funcionamento (5 Etapas)
$$\text{1. Entrada Normalizada } [0, 1] \;\longrightarrow\; \text{2. Forward Pass } (\text{ReLU} / \text{Softmax}) \;\longrightarrow\; \text{3. Perda } (\text{Cross-Entropy}) \;\longrightarrow\; \text{4. Backpropagation } (\nabla \mathbf{W}) \;\longrightarrow\; \text{5. Saída } (10 \text{ Probabilidades})$$

### Desafios Reais que Vamos Diagnosticar e Resolver na Prática
- **Variação na caligrafia:** Diferentes pessoas desenham o mesmo `"7"` (com ou sem traço horizontal) ou `"1"` (vertical puro, inclinado, ou com topo e base) de maneiras drasticamente distintas.
- **Overfitting (Sobreajuste):** Quando a rede "decora" os pixels exatos das imagens de treino e perde desempenho ao encontrar um traço inédito — problema que combateremos na prática com **`Dropout`** (desligamento aleatório de neurônios no treino) e **Aumento de Dados (*Data Augmentation*)** (pequenas rotações, translações e zoom sintéticos).

---

## Passo 1 — Configuração do Ambiente, Carregamento do `MNIST` e Auditoria dos Dados

Seguindo as boas práticas de Engenharia de Dados, antes de criar qualquer modelo precisamos carregar as bibliotecas, fixar as **sementes de números pseudoaleatórios** (*seeds* — para garantir que todo experimento seja 100% reprodutível) e inspecionar rigorosamente o **esquema dos dados**: dimensões (`.shape`), tipos numéricos (`.dtype`), faixa de valores e presença de falhas de leitura (`NaN` ou valores infinitos).

In [ ]:
# Biblioteca nativa para medir o tempo computacional (custo de treino e inferência)
import time

# Computação matricial e vetorial (a base matemática de todas as redes neurais)
import numpy as np

# Organização de tabelas comparativas de métricas
import pandas as pd

# Visualização científica de imagens, curvas de aprendizado e matrizes de confusão
import matplotlib.pyplot as plt
import seaborn as sns

# Ferramentas de divisão estratificada e métricas formais de classificação
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Framework industrial de Deep Learning (TensorFlow + Keras integrado)
import tensorflow as tf
from tensorflow import keras

# Exibe os gráficos diretamente abaixo de cada célula do Jupyter Notebook
%matplotlib inline

# Fixamos as sementes aleatórias para reprodutibilidade científica dos experimentos
np.random.seed(42)
tf.random.set_seed(42)

# Carrega o dataset clássico MNIST diretamente pelo Keras:
# (60.000 imagens de treino originais) e (10.000 imagens de teste de escritores distintos)
(X_train_full, y_train_full), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

# --- AUDITORIA DE ESQUEMA E INTEGRIDADE DOS SENSORES (PIXELS) ---
print("=== 1. DIMENSÕES E TIPOS DE DADOS (SCHEMA) ===")
print(f"Treino bruto (X_train_full): formato = {X_train_full.shape} | tipo = {X_train_full.dtype}")
print(f"Rótulos treino (y_train_full): formato = {y_train_full.shape} | tipo = {y_train_full.dtype}")
print(f"Teste bruto  (X_test_raw)  : formato = {X_test_raw.shape} | tipo = {X_test_raw.dtype}")
print(f"Rótulos teste  (y_test)      : formato = {y_test.shape} | tipo = {y_test.dtype}")

print("\n=== 2. VERIFICAÇÃO DE ESCALA RADIOMÉTRICA E DADOS AUSENTES (NaN / Inf) ===")
print(f"Faixa de intensidade dos pixels (Treino): mín = {X_train_full.min()} | máx = {X_train_full.max()}")
print(f"Faixa de intensidade dos pixels (Teste) : mín = {X_test_raw.min()} | máx = {X_test_raw.max()}")

# Verifica se há qualquer pixel corrompido (NaN = Not a Number ou Inf = Infinito)
nulos_treino = int(np.isnan(X_train_full).sum() + np.isinf(X_train_full).sum())
nulos_teste = int(np.isnan(X_test_raw).sum() + np.isinf(X_test_raw).sum())
nulos_rotulos = int(np.isnan(y_train_full).sum() + np.isnan(y_test).sum())

# Verifica se todos os rótulos pertencem estritamente ao conjunto {0, 1, 2, ..., 9}
classes_unicas = np.unique(y_train_full)
rotulos_invalidos = np.setdiff1d(classes_unicas, np.arange(10))

print(f"Pixels inválidos (NaN/Inf) -> Treino: {nulos_treino} | Teste: {nulos_teste}")
print(f"Rótulos ausentes: {nulos_rotulos} | Classes encontradas: {classes_unicas}")
print(f"Rótulos fora de [0..9]: {len(rotulos_invalidos)}")

### Interpretação da Auditoria do Conjunto de Dados

Os números acima revelam informações essenciais antes de qualquer modelagem:
1. **Estrutura Tensorial 3D `(60000, 28, 28)`:** Temos `60.000` matrizes empilhadas para aprendizado e `10.000` matrizes reservadas para teste final. Cada amostra possui `28` linhas (altura) por `28` colunas (largura).
2. **Codificação de 8 bits (`uint8` de `0` a `255`):** Em instrumentação óptica, um conversor analógico-digital (ADC) de 8 bits possui $2^8 = 256$ níveis discretos: `0` representa ausência de traço (fundo preto) e `255` representa intensidade máxima do traço manuscrito.
3. **Integridade Completa (`0` valores `NaN` ou inválidos):** Como o `MNIST` já passou por curadoria laboratorial (centralização por centro de massa em janelas $28 \times 28$), não há sensores falhos nem amostras sem rótulo — portanto, **não precisamos descartar nem imputar nenhuma linha**, preservando 100% das `70.000` imagens!

---

## Passo 2 — O que o Computador Enxerga vs. O Desafio da Variação na Caligrafia

Para um ser humano, reconhecer um número escrito em um pedaço de papel parece instantâneo. Para o computador, porém, **não existem "curvas" nem "algarismos" na memória**: existe apenas uma **matriz fria de $28 \times 28$ números inteiros**.

Além disso, surge aqui o nosso **1º Desafio Comum: a Variação na Caligrafia**:
- Algumas pessoas cortam o `"7"` com um traço horizontal no meio; outras escrevem apenas o ângulo agudo superior; outras inclinam a haste quase a $45^\circ$.
- O dígito `"1"` pode ser uma simples barra vertical, uma barra inclinada (quase idêntica à haste de um `"7"`!), ou ter "chapéu" no topo e base reta embaixo.

Vamos comprovar visualmente esses dois fatos inspecionando:
1. A **matriz numérica exata** pixel por pixel de uma única imagem;
2. Uma galeria comparando **10 caligrafias reais diferentes do `"7"`** e **10 caligrafias reais do `"1"`** presentes no conjunto de treino.

In [ ]:
# Seleciona o primeiro exemplo do dígito '7' no conjunto de treino para inspeção matricial
idx_sete = int(np.where(y_train_full == 7)[0][0])
imagem_exemplo = X_train_full[idx_sete]

# --- FIGURA 1: O QUE O COMPUTADOR REALMENTE ENXERGA (MATRIZ 28x28 DE INTEIROS) ---
fig, ax = plt.subplots(figsize=(8.5, 8.5))
ax.imshow(imagem_exemplo, cmap="gray", vmin=0, vmax=255)
ax.set_title(
    f"Anatomia Digital de uma Imagem 28x28 (Índice #{idx_sete} — Rótulo Real: {y_train_full[idx_sete]})\n"
    "Cada célula mostra a leitura inteira do pixel [0 = preto, 255 = branco]",
    fontsize=12,
    pad=12,
)

# Escreve o valor numérico exato dentro de cada pixel não nulo para abrir a "caixa-preta"
for linha in range(28):
    for coluna in range(28):
        valor_pixel = int(imagem_exemplo[linha, coluna])
        if valor_pixel > 0:
            # Escolhe texto escuro sobre fundo claro (>130) e texto claro sobre fundo escuro
            cor_texto = "black" if valor_pixel > 130 else "deepskyblue"
            ax.text(
                coluna,
                linha,
                str(valor_pixel),
                ha="center",
                va="center",
                color=cor_texto,
                fontsize=6.5,
                fontweight="bold",
            )

ax.set_xlabel("Coluna j (0 a 27)")
ax.set_ylabel("Linha i (0 a 27)")
plt.tight_layout()
plt.show()

# --- FIGURA 2: DESAFIO COMUM #1 — VARIAÇÃO REAL NA CALIGRAFIA HUMANA ('7' e '1') ---
indices_setes = np.where(y_train_full == 7)[0][:10]
indices_uns = np.where(y_train_full == 1)[0][:10]

fig, eixos = plt.subplots(2, 10, figsize=(14, 3.4))
fig.suptitle(
    "Desafio da Variação na Caligrafia: 10 Escritores Diferentes para o '7' (topo) e para o '1' (base)",
    fontsize=12,
    fontweight="bold",
)

for col in range(10):
    # Linha 0: 10 formas reais de escrever o número 7
    eixos[0, col].imshow(X_train_full[indices_setes[col]], cmap="gray")
    eixos[0, col].set_title(f"7 (#{col+1})", fontsize=9)
    eixos[0, col].axis("off")

    # Linha 1: 10 formas reais de escrever o número 1
    eixos[1, col].imshow(X_train_full[indices_uns[col]], cmap="gray")
    eixos[1, col].set_title(f"1 (#{col+1})", fontsize=9)
    eixos[1, col].axis("off")

plt.tight_layout()
plt.show()

### Análise das Imagens: Por que Regras Fixas (`if/else`) Falham Aqui?

Observe com atenção as duas figuras acima:
1. **Borda Suave (*Anti-aliasing*):** Os pixels não valem apenas `0` ou `255` — nas bordas do traço há valores intermediários (`45`, `118`, `204`), que capturam a espessura e a pressão da caneta.
2. **Incompatibilidade Pixel a Pixel:** Compare o `7 (#2)` com o `7 (#8)`, ou o `1 (#3)` inclinado à direita com o `1 (#6)` inclinado à esquerda. Se sobrepusermos duas matrizes de `"7"` diferentes, a maioria dos pixels altos (`> 200`) **não cai nas mesmas coordenadas $(i, j)$**! É exatamente por isso que programar regras fixas manuais é inviável e precisamos de Redes Neurais capazes de extrair padrões estatísticos.

---

## Passo 3 — Divisão Estrita (`Treino` / `Validação` / `Teste`) e Normalização (`[0, 1]`)

Antes de alimentar qualquer rede neural, aplicamos duas regras inegociáveis da Engenharia de Aprendizado de Máquina:

1. **Ordem Estrita de Divisão (*Strict Split Ordering* — Prevenção contra *Data Leakage*):**
   - O conjunto `Teste` (`10.000` imagens) deve permanecer trancado em um "cofre" até a avaliação final.
   - Para monitorar se a rede está **aprendendo de verdade** ou apenas **decorando** (*Overfitting*) época após época, dividimos as `60.000` imagens originais de treino em:
     - **Treino efetivo (`50.000` imagens):** Usado para calcular gradientes e ajustar os pesos $\mathbf{W}$.
     - **Validação (`10.000` imagens):** Usado ao fim de cada época como "simulado" para medir a capacidade de generalização em traços não usados no ajuste de pesos.
   - Fazemos essa divisão **antes** de qualquer transformação numérica e com amostragem **estratificada** (`stratify=y_train_full`), garantindo a mesma proporção de dígitos `0` a `9` em ambos os blocos.

2. **Normalização Radiométrica para `[0, 1]` ($\approx$ Adimensionalização):**
   - Como todo sensor óptico de 8 bits tem escala física conhecida no intervalo fixo $[0, 255]$, normalizamos cada conjunto dividindo por `255.0`:
     $$x_{\text{norm}}(i, j) = \frac{x_{\text{bruto}}(i, j)}{255{,}0} \in [0{,}0, \; 1{,}0]$$
   - **Por que isso é crucial?** Se passássemos valores como `250` diretamente na soma ponderada $u = \sum w_i x_i$, o potencial interno $u$ facilmente atingiria centenas ou milhares, explodindo exponenciais ($e^u$) na camada `Softmax` e desestabilizando o Gradiente Descendente!
   - **Formato 4D para a `CNN`:** Enquanto o `MLP` lê matrizes `(N, 28, 28)` e as achata, camadas convolucionais exigem explicitamente o eixo de **canais de cor** no final: `(N, 28, 28, 1)` (`1` canal = escala de cinza; se fosse uma foto colorida RGB, seriam `3` canais).

In [ ]:
# 1. DIVISÃO ESTRATIFICADA ANTES DE QUALQUER PRÉ-PROCESSAMENTO
# Reservamos 10.000 amostras para Validação e 50.000 para Treino efetivo
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=10000,
    stratify=y_train_full,  # Mantém exatamente a mesma proporção das classes 0..9
    random_state=42,
)

# 2. NORMALIZAÇÃO DE ENTRADA PARA [0.0, 1.0] EM PONTO FLUTUANTE DE 32 BITS (float32)
# float32 usa metade da memória do float64 e é o padrão nativo das GPUs e CPUs em Deep Learning
X_train = X_train_raw.astype("float32") / 255.0
X_val = X_val_raw.astype("float32") / 255.0
X_test = X_test_raw.astype("float32") / 255.0

# 3. ADIÇÃO DO EIXO DE CANAL (N, 28, 28, 1) PARA AS CAMADAS CONVOLUCIONAIS (CNN)
X_train_cnn = np.expand_dims(X_train, axis=-1)
X_val_cnn = np.expand_dims(X_val, axis=-1)
X_test_cnn = np.expand_dims(X_test, axis=-1)

print("=== RESUMO DAS PARTIÇÕES PREPARADAS ===")
print(f"Treino    -> MLP: {X_train.shape} | CNN: {X_train_cnn.shape} | escala: [{X_train.min():.1f}, {X_train.max():.1f}]")
print(f"Validação -> MLP: {X_val.shape} | CNN: {X_val_cnn.shape} | escala: [{X_val.min():.1f}, {X_val.max():.1f}]")
print(f"Teste     -> MLP: {X_test.shape} | CNN: {X_test_cnn.shape} | escala: [{X_test.min():.1f}, {X_test.max():.1f}]")

# 4. VERIFICAÇÃO DE BALANCEAMENTO DAS 10 CLASSES (0 a 9)
contagem_treino = np.bincount(y_train, minlength=10)
contagem_val = np.bincount(y_val, minlength=10)
contagem_teste = np.bincount(y_test, minlength=10)

fig, ax = plt.subplots(figsize=(10, 4))
digitos = np.arange(10)
largura = 0.27

ax.bar(digitos - largura, contagem_treino / len(y_train) * 100, width=largura, label="Treino (50k)", color="#2b5c8f")
ax.bar(digitos, contagem_val / len(y_val) * 100, width=largura, label="Validação (10k)", color="#38a3a5")
ax.bar(digitos + largura, contagem_teste / len(y_test) * 100, width=largura, label="Teste (10k)", color="#f4a261")

ax.axhline(10.0, color="crimson", linestyle="--", linewidth=1.2, label="Equilíbrio perfeito (10%)")
ax.set_xticks(digitos)
ax.set_xlabel("Classe (Dígito Manuscrito de 0 a 9)")
ax.set_ylabel("Participação no Conjunto (%)")
ax.set_title("Distribuição Percentual das Classes nos Conjuntos de Treino, Validação e Teste")
ax.set_ylim(0, 14)
ax.legend(ncol=4, loc="upper center")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Análise do Balanceamento das Classes

O gráfico confirma duas propriedades importantes:
- As três barras (Treino, Validação e Teste) são praticamente idênticas em cada dígito (~9% a ~11,2%, coladas na linha tracejada ideal de `10%`).
- Como **não há desbalanceamento severo** (nenhuma classe domina 90% da base), a **Acurácia Global** é uma métrica intuitiva e confiável — ainda assim, acompanharemos também o **F1-Score Macro** e a **Matriz de Confusão** para auditar cada dígito individualmente.

---

## Passo 4 — Caixa Aberta 1 (`NumPy` Puro): As 5 Etapas do Fluxo de Funcionamento por Dentro

Seguindo a metodologia deste repositório (*"Caixa Aberta $\rightarrow$ Ferramenta Industrial"*), antes de usar o `Keras` vamos abrir completamente o capô matemático e executar **uma iteração completa das 5 etapas do fluxo de reconhecimento** em `NumPy` puro sobre uma imagem real!

Veja a matemática exata de cada etapa:

1. **Entrada Normalizada + Achatamento (`Flatten` no MLP):**
   A matriz $28 \times 28$ é empilhada linha após linha em um único vetor coluna $\mathbf{x} \in [0, 1]^{784}$ ($28 \times 28 = 784$ entradas).

2. **Propagação Direta (*Forward Propagation*):**
   - **Nas camadas ocultas (`ReLU` — *Rectified Linear Unit*):** Calculamos $\mathbf{u}^{(1)} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$ e aplicamos um "diodo retificador ideal" que bloqueia sinais negativos e deixa passar sinais positivos sem saturar a derivada:
     $$\text{ReLU}(u) = \max(0, u) \qquad \Longrightarrow \qquad \text{ReLU}'(u) = \begin{cases} 1, & \text{se } u > 0 \\ 0, & \text{se } u \le 0 \end{cases}$$
   - **Na camada de saída (`Softmax` — 10 classes):** Calculamos os 10 potenciais brutos (*logits*) $\mathbf{z}^{(2)} = \mathbf{W}^{(2)}\mathbf{h}^{(1)} + \mathbf{b}^{(2)}$ e convertemos em **10 probabilidades estritamente positivas que somam `1,0` (`100%`)**:
     $$\hat{y}_c = \frac{e^{z_c}}{\sum_{j=0}^{9} e^{z_j}} \qquad \text{para } c \in \{0, 1, \dots, 9\}$$

3. **Função de Perda (*Loss Function — Cross-Entropy* / Entropia Cruzada):**
   Seja $\mathbf{d}$ o vetor *One-Hot* verdadeiro (ex.: para o dígito `3`, $\mathbf{d} = [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]$). A Entropia Cruzada mede o choque logarítmico da probabilidade atribuída à classe correta:
   $$L(\mathbf{d}, \hat{\mathbf{y}}) = -\sum_{c=0}^{9} d_c \ln(\hat{y}_c) = -\ln\big(\hat{y}_{\text{classe real}}\big)$$
   - Se a rede acerta com $\hat{y}_{\text{real}} = 0{,}99 \implies L = -\ln(0{,}99) \approx 0{,}010$ (erro quase nulo!).
   - Se a rede chuta às cegas ($10\%$ para cada dígito, $\hat{y}_{\text{real}} = 0{,}10$) $\implies L = -\ln(0{,}10) \approx 2{,}3026$.
   - Se a rede erra com alta confiança ($\hat{y}_{\text{real}} = 0{,}001$) $\implies L = -\ln(0{,}001) \approx 6{,}9078$ (penalidade pesadíssima!).

4. **Retropropagação (*Backpropagation*) + Gradiente Descendente:**
   Quando combinamos `Softmax` com `Cross-Entropy`, a Regra da Cadeia produz uma simplificação algébrica extraordinária: o gradiente do erro nos potenciais de saída $\mathbf{z}^{(2)}$ é simplesmente **a diferença direta entre o vetor previsto e o vetor real**!
   $$\frac{\partial L}{\partial \mathbf{z}^{(2)}} = \hat{\mathbf{y}} - \mathbf{d}$$
   A partir dele, propagamos o erro de volta até $\mathbf{W}^{(1)}$ e ajustamos todas as "válvulas" no sentido oposto ao gradiente ($\mathbf{W} \leftarrow \mathbf{W} - \eta \nabla_{\mathbf{W}} L$).

5. **Saída e Reconhecimento:**
   O dígito reconhecido é o índice de maior probabilidade: $\hat{c} = \arg\max_{c}(\hat{y}_c)$.

In [ ]:
# Funções matemáticas puras das ativações e da função de perda
def relu(u):
    # Diodo retificador: zera valores negativos e preserva positivos
    return np.maximum(0.0, u)


def derivada_relu(u):
    # Derivada vale 1 onde u > 0 e 0 onde u <= 0
    return (u > 0.0).astype(np.float32)


def softmax(z):
    # Subtrair max(z) antes da exponencial evita overflow numérico sem alterar o resultado!
    z_estavel = z - np.max(z)
    exp_z = np.exp(z_estavel)
    return exp_z / np.sum(exp_z)


def cross_entropy(probabilidades, classe_real):
    # np.clip protege contra log(0) caso alguma probabilidade tenda numericamente a zero
    prob_correta = np.clip(probabilidades[classe_real], 1e-12, 1.0)
    return -float(np.log(prob_correta))


# --- ETAPA 1: ENTRADA NORMALIZADA E ACHATAMENTO (FLATTEN) ---
imagem_amostra = X_train[0]              # Matriz 2D normalizada (28, 28) em [0, 1]
rotulo_real = int(y_train[0])            # Rótulo inteiro verdadeiro (0 a 9)
x_vetor = imagem_amostra.reshape(-1)     # Flatten: transforma (28, 28) -> vetor 1D (784,)

# Vetor alvo em codificação One-Hot (vale 1.0 apenas na posição do dígito verdadeiro)
d_onehot = np.zeros(10, dtype=np.float32)
d_onehot[rotulo_real] = 1.0

# Inicialização de He (padrão ideal para camadas ReLU) de uma mini-rede 784 -> 32 -> 10
rng = np.random.default_rng(42)
W1 = rng.normal(0.0, np.sqrt(2.0 / 784), size=(32, 784)).astype(np.float32)
b1 = np.zeros(32, dtype=np.float32)
W2 = rng.normal(0.0, np.sqrt(2.0 / 32), size=(10, 32)).astype(np.float32)
b2 = np.zeros(10, dtype=np.float32)

# --- ETAPA 2: PROPAGAÇÃO DIRETA (FORWARD PROPAGATION) ---
u1 = W1 @ x_vetor + b1                   # Potencial interno dos 32 neurônios ocultos
h1 = relu(u1)                            # Saída oculta após ReLU
z2 = W2 @ h1 + b2                        # Potenciais de saída (logits das 10 classes)
probs_inicial = softmax(z2)              # Vetor de 10 probabilidades que somam 1.0

# --- ETAPA 3: FUNÇÃO DE PERDA (CROSS-ENTROPY LOSS) ---
perda_inicial = cross_entropy(probs_inicial, rotulo_real)

# --- ETAPA 4: RETROPROPAGAÇÃO (BACKPROPAGATION) + GRADIENTE DESCENDENTE ---
eta = 0.05                               # Taxa de aprendizado (tamanho do passo)
dz2 = probs_inicial - d_onehot           # Gradiente elegante da saída (Softmax + Cross-Entropy)
dW2 = np.outer(dz2, h1)                  # Derivada parcial dL/dW2 (formato 10x32)
db2 = dz2                                # Derivada parcial dL/db2 (formato 10,)

dh1 = W2.T @ dz2                         # Erro retropropagado para os neurônios ocultos
du1 = dh1 * derivada_relu(u1)            # Multiplica pela inclinação local da ReLU
dW1 = np.outer(du1, x_vetor)             # Derivada parcial dL/dW1 (formato 32x784)
db1 = du1                                # Derivada parcial dL/db1 (formato 32,)

# Atualização dos pesos no sentido oposto ao gradiente (-eta * gradiente)
W2_novo = W2 - eta * dW2
b2_novo = b2 - eta * db2
W1_novo = W1 - eta * dW1
b1_novo = b1 - eta * db1

# --- ETAPA 5: NOVA SAÍDA APÓS 1 ÚNICO PASSO DE APRENDIZADO ---
probs_apos_1_passo = softmax(W2_novo @ relu(W1_novo @ x_vetor + b1_novo) + b2_novo)
perda_apos_1_passo = cross_entropy(probs_apos_1_passo, rotulo_real)

print(f"1. Entrada -> Matriz {imagem_amostra.shape} achatada para vetor {x_vetor.shape} | Dígito real = {rotulo_real}")
print(f"2. Soma das 10 probabilidades Softmax = {probs_inicial.sum():.6f} (100%)")
print(f"3. ANTES do Backpropagation -> P(dígito {rotulo_real}) = {probs_inicial[rotulo_real]*100:5.2f}% | Cross-Entropy Loss = {perda_inicial:.4f}")
print(f"4. APÓS 1 passo de Gradiente -> P(dígito {rotulo_real}) = {probs_apos_1_passo[rotulo_real]*100:5.2f}% | Cross-Entropy Loss = {perda_apos_1_passo:.4f}")
print(f"5. Vetor de probabilidades atualizado (0 a 9):\n   {np.round(probs_apos_1_passo, 3)}")

### O que Acabamos de Provar Numericamente?

Repare na beleza e transparência do resultado acima:
- Com pesos iniciais aleatórios, a rede chutava quase uniformemente entre os 10 dígitos, gerando uma perda *Cross-Entropy* próxima do valor teórico do chute cego ($-\ln(0{,}10) \approx 2{,}30$).
- Bastou **um único passo de *Backpropagation* + Gradiente Descendente** sobre essa amostra para a probabilidade atribuída ao dígito correto saltar expressivamente e a perda despencar! Quando repetimos esse mesmo cálculo em lotes (*mini-batches*) ao longo de `50.000` imagens variadas, a rede inteira converge.

---

## Passo 5 — Caixa Aberta 2 (`NumPy` Puro): Por que o `Flatten` Limita o `MLP` e Como a Convolução (`CNN`) Enxerga Formas

Antes de construir os dois modelos no `Keras`, precisamos entender visualmente a diferença física entre `MLP` e `CNN`:

### A Armadilha do `Flatten` no `MLP`
Imagine recortar uma fotografia $28 \times 28$ em 28 tiras horizontais finas e colar todas ponta a ponta formando uma fita comprida de `784` quadradinhos. Dois pixels que eram **vizinhos colados na vertical** — por exemplo, $(5, 14)$ e $(6, 14)$ — agora estão separados por `28` posições na fita! Para o `MLP`, a ordem dos 784 pixels nem importa: se embaralhássemos todas as imagens com a mesma permutação aleatória, o `MLP` aprenderia exatamente igual, pois ele **não sabe quem é vizinho de quem**.

### Como Funciona o Filtro Convolucional (`CNN`)
Em vez de destruir a grade 2D, uma **Camada Convolucional** desliza pequenas matrizes de pesos chamadas **filtros** ou ***kernels*** (tipicamente $3 \times 3$) sobre cada janela local da imagem, calculando a soma dos produtos elemento a elemento (**Convolução Discreta 2D**):

$$S(i, j) = \sum_{m=0}^{2} \sum_{n=0}^{2} I(i + m, \, j + n) \cdot K(m, n)$$

Logo após a convolução e a ativação `ReLU`, aplica-se o **`MaxPooling` $2 \times 2$** (*subamostragem por valor máximo*): a imagem é dividida em bloquinhos $2 \times 2$ e guarda-se apenas o maior sinal de cada bloco, reduzindo a dimensão pela metade (ex.: $26 \times 26 \rightarrow 13 \times 13$) e tornando a rede **tolerante a pequenos deslocamentos e tremores na escrita**!

Vamos implementar a **Convolução 2D** e o **`MaxPooling` em `NumPy` puro** (sem nenhuma caixa-preta!) e passar três filtros clássicos de Engenharia de Imagens (detector de bordas horizontais, bordas verticais e contornos) sobre um dígito real do `MNIST`:

In [ ]:
def convolucao_2d_manual(imagem, kernel):
    """Varre uma matriz 2D com um filtro (kernel) k_h x k_w em NumPy puro."""
    altura_img, largura_img = imagem.shape
    k_h, k_w = kernel.shape

    # Sem preenchimento extra nas bordas ('valid padding'), uma imagem 28x28 com filtro 3x3 gera 26x26
    altura_saida = altura_img - k_h + 1
    largura_saida = largura_img - k_w + 1
    mapa_caracteristicas = np.zeros((altura_saida, largura_saida), dtype=np.float32)

    # Desliza a janela 3x3 por todas as linhas (i) e colunas (j)
    for i in range(altura_saida):
        for j in range(largura_saida):
            janela_local = imagem[i : i + k_h, j : j + k_w]
            mapa_caracteristicas[i, j] = np.sum(janela_local * kernel)

    return mapa_caracteristicas


def max_pooling_2d_manual(mapa, pool_size=2):
    """Resume cada janela (pool_size x pool_size) pelo seu valor de pico máximo."""
    h, w = mapa.shape
    h_out, w_out = h // pool_size, w // pool_size
    mapa_reduzido = np.zeros((h_out, w_out), dtype=np.float32)

    for i in range(h_out):
        for j in range(w_out):
            bloco = mapa[
                i * pool_size : (i + 1) * pool_size,
                j * pool_size : (j + 1) * pool_size,
            ]
            mapa_reduzido[i, j] = np.max(bloco)

    return mapa_reduzido


# Escolhemos um dígito '8' real (rico em curvas horizontais, verticais e laços!)
idx_oito = int(np.where(y_train == 8)[0][0])
img_oito = X_train[idx_oito]

# Três filtros 3x3 clássicos de Processamento Digital de Imagens (Sobel / Laplaciano)
filtro_borda_horizontal = np.array(
    [[ 1.0,  2.0,  1.0],
     [ 0.0,  0.0,  0.0],
     [-1.0, -2.0, -1.0]], dtype=np.float32
)

filtro_borda_vertical = np.array(
    [[ 1.0,  0.0, -1.0],
     [ 2.0,  0.0, -2.0],
     [ 1.0,  0.0, -1.0]], dtype=np.float32
)

filtro_contorno = np.array(
    [[-1.0, -1.0, -1.0],
     [-1.0,  8.0, -1.0],
     [-1.0, -1.0, -1.0]], dtype=np.float32
)

# Aplica Convolução + ReLU (mantendo apenas respostas positivas do detector)
mapa_horiz = relu(convolucao_2d_manual(img_oito, filtro_borda_horizontal))
mapa_vert = relu(convolucao_2d_manual(img_oito, filtro_borda_vertical))
mapa_contorno = relu(convolucao_2d_manual(img_oito, filtro_contorno))

# Aplica MaxPooling 2x2 sobre o mapa de contornos (26x26 -> 13x13)
mapa_pooled = max_pooling_2d_manual(mapa_contorno, pool_size=2)

# Visualização lado a lado da extração espacial de características
fig, eixos = plt.subplots(1, 5, figsize=(15, 3.3))
paineis = [
    (img_oito, f"1. Original\n({img_oito.shape[0]}x{img_oito.shape[1]})", "gray"),
    (mapa_horiz, f"2. Filtro Horizontal\n({mapa_horiz.shape[0]}x{mapa_horiz.shape[1]}) + ReLU", "magma"),
    (mapa_vert, f"3. Filtro Vertical\n({mapa_vert.shape[0]}x{mapa_vert.shape[1]}) + ReLU", "magma"),
    (mapa_contorno, f"4. Filtro Contorno\n({mapa_contorno.shape[0]}x{mapa_contorno.shape[1]}) + ReLU", "magma"),
    (mapa_pooled, f"5. Após MaxPool 2x2\n({mapa_pooled.shape[0]}x{mapa_pooled.shape[1]}) — 4x menor!", "magma"),
]

for ax, (matriz, titulo, paleta) in zip(eixos, paineis):
    ax.imshow(matriz, cmap=paleta)
    ax.set_title(titulo, fontsize=9.5)
    ax.axis("off")

plt.suptitle("Caixa Aberta: Como Filtros Convolucionais 3x3 Decompõem a Geometria de um Dígito", fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

### A Grande Sacada da `CNN`: Filtros Aprendidos Automaticamente!

Olhe para os painéis 2, 3, 4 e 5 acima:
- O **Filtro Horizontal** acendeu apenas o topo e a base das curvas do `"8"`.
- O **Filtro Vertical** acendeu apenas as paredes laterais esquerda e direita do `"8"`.
- O **`MaxPooling` $2 \times 2$** comprimiu o mapa de $26 \times 26 = 676$ números para $13 \times 13 = 169$ números (**redução de 75% nos dados**), preservando nítida a silhueta geométrica do `"8"`!

Na Engenharia Clássica dos anos 1980/1990, engenheiros precisavam projetar esses filtros $3 \times 3$ manualmente para cada problema. Na **`CNN` moderna**, colocamos dezenas de matrizes $3 \times 3$ inicialmente aleatórias e deixamos o próprio **`Backpropagation` descobrir sozinho** quais filtros separam melhor os dígitos `0` a `9`!

---

## Passo 6 — Construindo e Treinando o Modelo 1: `MLP` (Rede Totalmente Conectada)

Vamos agora construir nosso primeiro modelo completo em `TensorFlow/Keras`: o **Perceptron Multicamadas (`MLP`)**.

### Arquitetura Escolhida (`784 - 128 - 64 - 10`)
1. **`Input(shape=(28, 28))` + `Flatten()`:** Achata cada matriz $28 \times 28$ em um vetor de `784` números.
2. **`Dense(128, activation="relu")`:** 1ª camada oculta totalmente conectada ($784 \times 128 + 128 = 100.480$ parâmetros!).
3. **`Dense(64, activation="relu")`:** 2ª camada oculta ($128 \times 64 + 64 = 8.256$ parâmetros).
4. **`Dense(10, activation="softmax")`:** Camada de saída com 10 neurônios ($64 \times 10 + 10 = 650$ parâmetros), entregando as 10 probabilidades.

> **Nota Prática sobre `sparse_categorical_crossentropy`:** No `Keras`, quando nossos rótulos `y_train` já estão guardados como números inteiros simples (`0, 1, ..., 9`) em vez de vetores *One-Hot* (`[0,0,1,...]`), usamos `loss="sparse_categorical_crossentropy"`. Matematicamente, ela calcula **exatamente a mesma Entropia Cruzada** $-\ln(\hat{y}_{\text{correto}})$ que implementamos na Caixa Aberta, economizando memória!

In [ ]:
tf.random.set_seed(42)

# Construção sequencial do MLP (Rede Neural Totalmente Conectada)
modelo_mlp = keras.Sequential(
    [
        keras.Input(shape=(28, 28), name="entrada_28x28"),
        keras.layers.Flatten(name="achatamento_784"),
        keras.layers.Dense(128, activation="relu", name="oculta_densa_1"),
        keras.layers.Dense(64, activation="relu", name="oculta_densa_2"),
        keras.layers.Dense(10, activation="softmax", name="saida_10_classes"),
    ],
    name="MLP_Totalmente_Conectado",
)

# Exibe o resumo da topologia e a contagem exata de parâmetros por camada
modelo_mlp.summary()

# Compila o modelo com otimizador Adam (Gradiente Descendente com taxa adaptativa + Momentum)
modelo_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treina por 12 épocas em mini-batches de 128 imagens, monitorando a Validação a cada época
inicio_mlp = time.perf_counter()
historico_mlp = modelo_mlp.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=128,
    verbose=1,
)
tempo_treino_mlp = time.perf_counter() - inicio_mlp
print(f"\nTempo total de treinamento do MLP: {tempo_treino_mlp:.2f} segundos")

# --- GRÁFICO DAS CURVAS DE APRENDIZADO DO MLP (DIAGNÓSTICO DE OVERFITTING) ---
epocas_mlp = range(1, len(historico_mlp.history["loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Painel 1: Evolução da Perda (Cross-Entropy)
ax1.plot(epocas_mlp, historico_mlp.history["loss"], "o-", label="Treino", color="#1d3557")
ax1.plot(epocas_mlp, historico_mlp.history["val_loss"], "s--", label="Validação", color="#e63946")
ax1.set_title("MLP — Evolução da Função de Perda (Cross-Entropy)")
ax1.set_xlabel("Época")
ax1.set_ylabel("Perda (Menor é melhor)")
ax1.legend()
ax1.grid(alpha=0.3)

# Painel 2: Evolução da Acurácia (%)
ax2.plot(epocas_mlp, [a * 100 for a in historico_mlp.history["accuracy"]], "o-", label="Treino", color="#1d3557")
ax2.plot(epocas_mlp, [a * 100 for a in historico_mlp.history["val_accuracy"]], "s--", label="Validação", color="#e63946")
ax2.set_title("MLP — Evolução da Acurácia (%)")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia (%)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Diagnóstico de Engenharia nas Curvas do `MLP`: O Sintoma Clássico do *Overfitting*

Observe atentamente o comportamento das duas curvas (azul = Treino, vermelha = Validação) a partir da **5ª ou 6ª época**:
1. **Na curva azul (Treino):** A perda cai continuamente rumo a zero e a acurácia passa de **99,4%** — a rede está praticamente decorando cada uma das `50.000` matrizes de treino!
2. **Na curva vermelha (Validação):** A acurácia estaciona perto de **97,7%** e a **perda de validação (`val_loss`) para de cair e começa a subir levemente**!
3. **Causa física:** Apenas a primeira camada `Flatten(784) -> Dense(128)` concentra **`100.480` pesos** (`91,8%` de toda a rede!) amarrados rigidamente às coordenadas $(i, j)$ fixas.

### Provando na Prática a Limitação Espacial do `MLP`
No mundo real (leitura de formulários, câmeras em esteiras industriais), o objeto nem sempre está perfeitamente cravado no centro milimétrico do quadro! O que acontece com nosso `MLP` recém-treinado se pegarmos as imagens de teste e apenas **deslocarmos o dígito `3` pixels para a direita e `3` pixels para baixo** (sem distorcer nada do formato do número)? Vamos fazer o teste experimental agora!

In [ ]:
def deslocar_imagens_2d(imagens, dy=3, dx=3):
    """
    Translada imagens em dy linhas (vertical) e dx colunas (horizontal),
    preenchendo as margens novas com fundo preto (0.0), sem 'dar a volta' nas bordas.
    Funciona tanto para arrays 3D (N, 28, 28) quanto 4D (N, 28, 28, 1).
    """
    deslocadas = np.zeros_like(imagens)
    h, w = imagens.shape[1], imagens.shape[2]

    orig_y0, orig_y1 = max(0, -dy), min(h, h - dy)
    dest_y0, dest_y1 = max(0, dy), min(h, h + dy)

    orig_x0, orig_x1 = max(0, -dx), min(w, w - dx)
    dest_x0, dest_x1 = max(0, dx), min(w, w + dx)

    deslocadas[:, dest_y0:dest_y1, dest_x0:dest_x1] = imagens[
        :, orig_y0:orig_y1, orig_x0:orig_x1
    ]
    return deslocadas


# Cria uma cópia do conjunto de teste deslocada apenas 3 pixels para baixo e 3 para a direita
X_test_deslocado = deslocar_imagens_2d(X_test, dy=3, dx=3)

# Avalia o MLP nos dois cenários: centralizado original vs. levemente deslocado
_, acc_mlp_central = modelo_mlp.evaluate(X_test, y_test, verbose=0)
_, acc_mlp_deslocado = modelo_mlp.evaluate(X_test_deslocado, y_test, verbose=0)

print("=== TESTE DE SENSIBILIDADE ESPACIAL DO MLP ===")
print(f"Acurácia do MLP no Teste Centralizado Original : {acc_mlp_central * 100:.2f}%")
print(f"Acurácia do MLP com Deslocamento de (3px, 3px) : {acc_mlp_deslocado * 100:.2f}%")
print(f"Queda abrupta de desempenho                    : -{(acc_mlp_central - acc_mlp_deslocado) * 100:.2f} pontos percentuais!")

# Visualiza 5 exemplos reais mostrando como o número continua legível, mas o MLP se perde!
probs_orig = modelo_mlp.predict(X_test[:5], verbose=0)
probs_desl = modelo_mlp.predict(X_test_deslocado[:5], verbose=0)

fig, eixos = plt.subplots(2, 5, figsize=(13, 5.2))
for col in range(5):
    pred_o = int(np.argmax(probs_orig[col]))
    pred_d = int(np.argmax(probs_desl[col]))
    real = int(y_test[col])

    eixos[0, col].imshow(X_test[col], cmap="gray")
    eixos[0, col].set_title(f"Original (Real: {real})\nMLP prevê: {pred_o}", fontsize=9.5, color="darkgreen" if pred_o == real else "crimson")
    eixos[0, col].axis("off")

    eixos[1, col].imshow(X_test_deslocado[col], cmap="gray")
    eixos[1, col].set_title(f"Deslocado +3px (Real: {real})\nMLP prevê: {pred_d}", fontsize=9.5, color="darkgreen" if pred_d == real else "crimson")
    eixos[1, col].axis("off")

plt.suptitle("Por que o Flatten Limita o MLP: Apenas 3 Pixels de Deslocamento Enganam os Pesos Fixos!", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Entendendo o Colapso Espacial do `MLP`

O experimento acima torna palpável a limitação teórica do `MLP`: para qualquer pessoa na sala, os dígitos da linha inferior são **exatamente os mesmos números** da linha superior! Porém, ao deslocar o desenho em `(3, 3)` pixels, no vetor achatado de `784` entradas isso equivale a deslocar todos os sinais em $3 \times 28 + 3 = 87$ colunas de distância — caindo em conexões sinápticas treinadas para outras regiões!

---

## Passo 7 — Modelo 2: `CNN` (Rede Neural Convolucional) + Combate ao *Overfitting* (`Dropout` e `Data Augmentation`)

Vamos agora construir a solução de nível industrial que resolve simultaneamente a **cegueira espacial**, a **variação na caligrafia** e o ***Overfitting***:

### As 4 Camadas de Engenharia da Nossa `CNN`
1. **Aumento de Dados em Tempo Real (*Data Augmentation*):**
   Inserimos logo na entrada três camadas de perturbação estocástica (`RandomRotation(0.05)`, `RandomTranslation(0.08, 0.08)` e `RandomZoom(0.08)`).
   - **Como funciona:** A cada época de treino, cada imagem sofre pequenas rotações aleatórias ($\approx \pm 18^\circ$), translações ($\pm 8\% \approx \pm 2{,}2$ pixels) e variações de escala ($\pm 8\%$). A rede nunca vê exatamente a mesma matriz rígida duas vezes, sendo forçada a aprender a **essência geométrica** dos traços!
   - **Detalhe importante no `Keras`:** Essas camadas ligam automaticamente durante `fit()` (`training=True`) e **desligam sozinhas** durante `evaluate()` e `predict()` (`training=False`), avaliando as imagens de teste sem distorções!
2. **1º Bloco Convolucional (`Conv2D(32, 3x3)` + `MaxPooling2D(2x2)`):**
   `32` filtros $3 \times 3$ varrem a imagem $28 \times 28 \times 1$ gerando `32` mapas $26 \times 26$ de bordas e retas; o `MaxPooling` resume para $13 \times 13 \times 32$.
3. **2º Bloco Convolucional (`Conv2D(64, 3x3)` + `MaxPooling2D(2x2)`):**
   `64` filtros $3 \times 3$ combinam as bordas anteriores em formas mais complexas (laços fechados do `0`/`8`/`9`, cruzamentos do `4`/`7`); o `MaxPooling` resume para $5 \times 5 \times 64$.
4. **Classificador Denso com `Dropout(0.4)` + Parada Antecipada (*Early Stopping*):**
   - Só agora, depois que a geometria 2D já foi destilada em `64` mapas compactos $5 \times 5$, aplicamos `Flatten()` ($5 \times 5 \times 64 = 1.600$ características geométricas de alto nível).
   - **`Dropout(0.4)` ($\approx$ Ensaio de Redundância Tolerante a Falhas):** Em cada lote de treino, `40%` dos neurônios são zerados aleatoriamente, impedindo que neurônios vizinhos "combinem de decorar" um ruído específico da base.
   - **`EarlyStopping`:** Monitora `val_loss` e restaura automaticamente os melhores pesos caso a validação pare de evoluir.

In [ ]:
tf.random.set_seed(42)

# Construção da CNN completa com Data Augmentation e Dropout integrados
modelo_cnn = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1), name="entrada_imagem_28x28x1"),
        # --- Combate à Variação de Caligrafia e ao Overfitting: Data Augmentation ---
        keras.layers.RandomRotation(0.05, name="aug_rotacao"),
        keras.layers.RandomTranslation(0.08, 0.08, name="aug_translacao"),
        keras.layers.RandomZoom(0.08, name="aug_zoom"),
        # --- 1º Estágio de Extração Espacial (Bordas, Cantos e Inclinações) ---
        keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu", name="conv1"),
        keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool1"),
        # --- 2º Estágio de Extração Espacial (Arcos, Laços e Partes do Dígito) ---
        keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu", name="conv2"),
        keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool2"),
        # --- Classificação Final com Regularização por Dropout ---
        keras.layers.Flatten(name="achatamento_mapas"),
        keras.layers.Dropout(0.4, name="dropout_40pct"),
        keras.layers.Dense(64, activation="relu", name="oculta_densa"),
        keras.layers.Dense(10, activation="softmax", name="saida_10_classes"),
    ],
    name="CNN_Convolucional_Robusta",
)

modelo_cnn.summary()

modelo_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Callback de segurança: restaura os pesos da melhor época de validação se parar de melhorar
parada_antecipada = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

inicio_cnn = time.perf_counter()
historico_cnn = modelo_cnn.fit(
    X_train_cnn,
    y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[parada_antecipada],
    verbose=1,
)
tempo_treino_cnn = time.perf_counter() - inicio_cnn
print(f"\nTempo total de treinamento da CNN: {tempo_treino_cnn:.2f} segundos")

# --- GRÁFICO DAS CURVAS DE APRENDIZADO DA CNN ---
epocas_cnn = range(1, len(historico_cnn.history["loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(epocas_cnn, historico_cnn.history["loss"], "o-", label="Treino (com Dropout + Augmentation)", color="#2a9d8f")
ax1.plot(epocas_cnn, historico_cnn.history["val_loss"], "s--", label="Validação (100% da rede ativa)", color="#e76f51")
ax1.set_title("CNN — Evolução da Função de Perda (Cross-Entropy)")
ax1.set_xlabel("Época")
ax1.set_ylabel("Perda (Menor é melhor)")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epocas_cnn, [a * 100 for a in historico_cnn.history["accuracy"]], "o-", label="Treino (com Dropout + Augmentation)", color="#2a9d8f")
ax2.plot(epocas_cnn, [a * 100 for a in historico_cnn.history["val_accuracy"]], "s--", label="Validação (100% da rede ativa)", color="#e76f51")
ax2.set_title("CNN — Evolução da Acurácia (%)")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia (%)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Por que na `CNN` a Curva de Validação Fica *Melhor* que a de Treino?!

Quem vê esse gráfico pela primeira vez costuma estranhar: *"No `MLP` o treino era melhor que a validação; por que na `CNN` a validação (laranja) tem perda menor e acurácia maior (~99%) do que o próprio treino (verde)?"*

A resposta é uma excelente analogia de Engenharia e Esporte de Alto Rendimento:
- **Durante o Treino (`training=True`):** A `CNN` está "correndo na areia fofa usando um colete de chumbo" — as imagens estão sendo giradas, deslocadas e ampliadas pelo *Data Augmentation*, **e** `40%` dos neurônios estão desligados pelo `Dropout`!
- **Durante a Validação e o Teste (`training=False`):** O colete de chumbo é retirado — não há distorções artificiais e **100% dos neurônios operam juntos em sinergia**!
- Além disso, repare no `summary()`: a `CNN` inteira tem apenas **`121.930` parâmetros** (praticamente o mesmo tamanho do `MLP`, que tinha `109.386`!), mas na 1ª camada convolucional (`conv1`) ela precisou de apenas **`320` pesos** em vez de `100.480`, graças ao **compartilhamento espacial de filtros**!

---

## Passo 8 — Comparação Final no Conjunto de Teste Inédito (`10.000` Imagens) e Matrizes de Confusão

Chegou a hora de abrir o "cofre" das `10.000` imagens de teste nunca vistas nem no treino nem na validação e comparar lado a lado **`MLP` vs. `CNN`** sob quatro critérios de engenharia:
1. Poder preditivo padrão (**Acurácia** e **F1-Score Macro**);
2. Robustez geométrica (repetindo exatamente o **teste de deslocamento de `(3px, 3px)`** na `CNN`!);
3. Custo operacional (quantidade de parâmetros e latência de inferência);
4. Diagnóstico fino de erros entre pares de dígitos via **Matrizes de Confusão**.

In [ ]:
# 1. Previsões e medição de latência de inferência nas 10.000 imagens inéditas de Teste
t0 = time.perf_counter()
probs_teste_mlp = modelo_mlp.predict(X_test, verbose=0)
latencia_mlp = time.perf_counter() - t0
y_pred_mlp = np.argmax(probs_teste_mlp, axis=1)

t0 = time.perf_counter()
probs_teste_cnn = modelo_cnn.predict(X_test_cnn, verbose=0)
latencia_cnn = time.perf_counter() - t0
y_pred_cnn = np.argmax(probs_teste_cnn, axis=1)

# 2. Repetição do Teste de Deslocamento Espacial (+3px vertical, +3px horizontal) na CNN
X_test_cnn_deslocado = deslocar_imagens_2d(X_test_cnn, dy=3, dx=3)
y_pred_mlp_desl = np.argmax(modelo_mlp.predict(X_test_deslocado, verbose=0), axis=1)
y_pred_cnn_desl = np.argmax(modelo_cnn.predict(X_test_cnn_deslocado, verbose=0), axis=1)

# 3. Tabela Comparativa Consolidada (MLP vs. CNN)
tabela_comparativa = pd.DataFrame(
    {
        "Modelo": ["MLP (Totalmente Conectado)", "CNN (Convolucional + Dropout + Aug)"],
        "Acurácia Teste (%)": [
            round(accuracy_score(y_test, y_pred_mlp) * 100, 2),
            round(accuracy_score(y_test, y_pred_cnn) * 100, 2),
        ],
        "F1-Score Macro (%)": [
            round(f1_score(y_test, y_pred_mlp, average="macro") * 100, 2),
            round(f1_score(y_test, y_pred_cnn, average="macro") * 100, 2),
        ],
        "Acurácia Deslocado +3px (%)": [
            round(accuracy_score(y_test, y_pred_mlp_desl) * 100, 2),
            round(accuracy_score(y_test, y_pred_cnn_desl) * 100, 2),
        ],
        "Erros em 10.000 Imagens": [
            int(np.sum(y_pred_mlp != y_test)),
            int(np.sum(y_pred_cnn != y_test)),
        ],
        "Parâmetros Treináveis": [
            modelo_mlp.count_params(),
            modelo_cnn.count_params(),
        ],
        "Inferência 10k imgs (s)": [
            round(latencia_mlp, 3),
            round(latencia_cnn, 3),
        ],
    }
)

print("=== TABELA COMPARATIVA FINAL NO CONJUNTO DE TESTE INÉDITO (N = 10.000) ===")
print(tabela_comparativa.to_string(index=False))

print("\n=== RELATÓRIO POR CLASSE (PRECISÃO, RECALL E F1-SCORE) — MODELO CNN ===")
print(classification_report(y_test, y_pred_cnn, digits=4))

# 4. Matrizes de Confusão Lado a Lado (MLP vs. CNN)
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
cm_cnn = confusion_matrix(y_test, y_pred_cnn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(cm_mlp, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax1)
ax1.set_title(f"Matriz de Confusão — MLP\n(Total de erros: {int(np.sum(y_pred_mlp != y_test))} em 10.000)")
ax1.set_xlabel("Dígito Previsto pelo MLP")
ax1.set_ylabel("Dígito Real (Verdadeiro)")

sns.heatmap(cm_cnn, annot=True, fmt="d", cmap="Greens", cbar=False, ax=ax2)
ax2.set_title(f"Matriz de Confusão — CNN\n(Total de erros: {int(np.sum(y_pred_cnn != y_test))} em 10.000)")
ax2.set_xlabel("Dígito Previsto pela CNN")
ax2.set_ylabel("Dígito Real (Verdadeiro)")

plt.tight_layout()
plt.show()

### Leitura de Engenharia da Comparação `MLP` vs. `CNN`

Os números acima sintetizam por que as Redes Neurais Convolucionais revolucionaram a Visão Computacional:
1. **Corte Drástico na Taxa de Erro:** Passar de ~97,7% (`MLP`, cerca de `220–240` erros em `10.000`) para ~99,0%+ (`CNN`, menos de `100` erros em `10.000`) representa **eliminar mais da metade de todas as falhas de leitura**! Em uma triagem postal automática processando `1.000.000` de encomendas por dia, isso evita mais de **`12.000` pacotes extraviados diariamente**.
2. **Imunidade ao Deslocamento Espacial:** Enquanto o `MLP` perde dezenas de pontos percentuais quando o dígito é escrito `3` pixels fora de centro, a `CNN` (combinando janelas convolucionais, `MaxPooling` e *Data Augmentation*) mantém acurácia altíssima mesmo sob translação!
3. **O que revelam os valores fora da diagonal principal da Matriz de Confusão?**
   - Os erros remanescentes concentram-se exatamente nos pares onde a própria caligrafia humana é ambígua: **`4` confundido com `9`** (quando o topo do `4` se fecha), **`7` confundido com `2` ou `1`**, e **`5` confundido com `3`**.
   - Em contrapartida, há um *trade-off* operacional claro: como varrer filtros 2D exige mais multiplicações por imagem do que multiplicar vetores 1D, o tempo de treino e inferência da `CNN` em CPU é maior que o do `MLP` (em placas gráficas / GPUs industriais, porém, ambas rodam em milissegundos!).

---

## Passo 9 — Auditando o Vetor de Probabilidades `Softmax`, os Erros Reais e o Interior da `CNN`

Para fechar nosso diagnóstico prático, vamos responder a duas perguntas fundamentais:
1. **Como são as imagens em que até uma `CNN` de ~99% erra?** E qual foi o vetor de 10 probabilidades `Softmax` nesses casos?
2. **O que os `32` filtros $3 \times 3$ da camada `conv1` realmente aprenderam a enxergar** depois do treinamento com *Backpropagation*?

In [ ]:
# Separa índices de acertos e erros reais da CNN no conjunto de teste
indices_acertos = np.where(y_pred_cnn == y_test)[0]
indices_erros = np.where(y_pred_cnn != y_test)[0]

# --- FIGURA 1: VETOR DE PROBABILIDADES SOFTMAX EM 3 ACERTOS E 3 ERROS REAIS ---
fig, eixos = plt.subplots(2, 6, figsize=(16, 5.2))
fig.suptitle(
    "Inspeção das Saídas Softmax (10 Classes): Linha Superior = Acertos | Linha Inferior = Caligrafias Ambíguas (Erros)",
    fontsize=12,
    fontweight="bold",
)

for k in range(3):
    # --- Linha 0: Acertos da CNN ---
    idx_ok = int(indices_acertos[k])
    eixos[0, 2 * k].imshow(X_test[idx_ok], cmap="gray")
    eixos[0, 2 * k].set_title(f"Real: {y_test[idx_ok]} | Prev: {y_pred_cnn[idx_ok]}\nConf: {probs_teste_cnn[idx_ok].max()*100:.1f}%", fontsize=9, color="darkgreen")
    eixos[0, 2 * k].axis("off")

    cores_ok = ["#2a9d8f" if c == y_test[idx_ok] else "#cbd5e1" for c in range(10)]
    eixos[0, 2 * k + 1].bar(range(10), probs_teste_cnn[idx_ok] * 100, color=cores_ok)
    eixos[0, 2 * k + 1].set_xticks(range(10))
    eixos[0, 2 * k + 1].set_ylim(0, 105)
    eixos[0, 2 * k + 1].set_title("Softmax (%)", fontsize=9)
    eixos[0, 2 * k + 1].grid(axis="y", alpha=0.3)

    # --- Linha 1: Erros Reais da CNN (Variação Extrema de Caligrafia) ---
    if k < len(indices_erros):
        idx_err = int(indices_erros[k])
        eixos[1, 2 * k].imshow(X_test[idx_err], cmap="gray")
        eixos[1, 2 * k].set_title(f"Real: {y_test[idx_err]} | Prev: {y_pred_cnn[idx_err]}\nConf: {probs_teste_cnn[idx_err].max()*100:.1f}%", fontsize=9, color="crimson")
        eixos[1, 2 * k].axis("off")

        cores_err = []
        for c in range(10):
            if c == y_test[idx_err]:
                cores_err.append("#2a9d8f")   # Verde = classe verdadeira
            elif c == y_pred_cnn[idx_err]:
                cores_err.append("#e63946")   # Vermelho = chute errado da rede
            else:
                cores_err.append("#cbd5e1")
        eixos[1, 2 * k + 1].bar(range(10), probs_teste_cnn[idx_err] * 100, color=cores_err)
        eixos[1, 2 * k + 1].set_xticks(range(10))
        eixos[1, 2 * k + 1].set_ylim(0, 105)
        eixos[1, 2 * k + 1].set_title("Verde=Real | Verm.=Prev", fontsize=8.5)
        eixos[1, 2 * k + 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# --- FIGURA 2: ABRINDO A CNN TREINADA — OS MAPAS DE CARACTERÍSTICAS (FEATURE MAPS) APRENDIDOS ---
# Passamos 1 imagem de teste diretamente pela 1ª camada convolucional treinada ('conv1')
amostra_teste_4d = X_test_cnn[0:1]  # Formato (1, 28, 28, 1)
mapas_aprendidos = modelo_cnn.get_layer("conv1")(amostra_teste_4d).numpy()[0]  # Formato (26, 26, 32)

fig, eixos = plt.subplots(1, 9, figsize=(16, 2.5))
eixos[0].imshow(X_test[0], cmap="gray")
eixos[0].set_title(f"Entrada\n(Dígito {y_test[0]})", fontsize=9.5, fontweight="bold")
eixos[0].axis("off")

for f_idx in range(8):
    eixos[f_idx + 1].imshow(mapas_aprendidos[:, :, f_idx], cmap="viridis")
    eixos[f_idx + 1].set_title(f"Filtro #{f_idx + 1}\n(26x26)", fontsize=9)
    eixos[f_idx + 1].axis("off")

plt.suptitle(
    "O que a 1ª Camada Convolucional ('conv1') Aprendeu Sozinha via Backpropagation (8 dos 32 Filtros)",
    fontsize=11.5,
    fontweight="bold",
    y=1.05,
)
plt.tight_layout()
plt.show()

### O que Essas Duas Figuras Nos Ensinam?

1. **Transparência das Probabilidades `Softmax`:** Nos acertos normais (linha superior), a barra verde crava quase **100%** na classe certa e **0%** nas outras 9. Já nos erros de caligrafia extrema (linha inferior — traços cortados, borrados ou incompletos), o gráfico de barras mostra que **a própria rede frequentemente divide sua probabilidade entre duas hipóteses plausíveis** (barra vermelha vs. barra verde)! Na Engenharia, essa medida de incerteza vale ouro para sistemas críticos.
2. **Filtros Descobertos Automaticamente:** Na segunda figura, nenhum programador escreveu matrizes de Sobel na `CNN` — mesmo assim, cada um dos filtros aprendidos na camada `conv1` especializou-se espontaneamente em uma orientação geométrica diferente (diagonais superiores, terminações inferiores, bordas esquerdas ou o fundo negativo do traço)!

---

## 10. Exercícios Práticos de Fixação

Agora é a sua vez de consolidar o domínio prático! Resolva os 3 desafios nas células abaixo antes de abrir os gabaritos comentados.

### Exercício 1: Isolando e Provando o Efeito do `Dropout` contra o *Overfitting*
Em muitos projetos industriais de Engenharia, **não temos `50.000` fotos rotuladas**, mas apenas algumas centenas ou poucos milhares — cenário onde o *Overfitting* é severo!
1. Recorte apenas as primeiras **`1.500` imagens de treino** (`X_peq = X_train[:1500]`, `y_peq = y_train[:1500]`).
2. Treine por `25` épocas (`batch_size=64`, `verbose=0`) dois modelos `MLP` com uma camada oculta de `256` neurônios (`relu`), avaliando em `(X_val, y_val)`:
   - `mlp_sem_dropout`: sem regularização;
   - `mlp_com_dropout`: adicionando `keras.layers.Dropout(0.5)` logo após a camada `Dense(256, activation="relu")`.
3. Imprima a acurácia de **Treino** e de **Validação** na última época dos dois modelos e compare o *gap* (diferença `Treino - Validação`, que mede o grau de memorização).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado do Exercício 1</b></summary>

```python
tf.random.set_seed(42)

# Subconjunto pequeno (cenário típico de bancada industrial com poucas imagens coletadas)
X_peq = X_train[:1500]
y_peq = y_train[:1500]

# Modelo A: Sem Dropout (livre para decorar os 1.500 exemplos)
mlp_sem_dropout = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
mlp_sem_dropout.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_sem = mlp_sem_dropout.fit(X_peq, y_peq, validation_data=(X_val, y_val), epochs=25, batch_size=64, verbose=0)

# Modelo B: Com Dropout de 50% (desliga metade dos neurônios a cada passo de treino)
mlp_com_dropout = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation="softmax"),
])
mlp_com_dropout.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_com = mlp_com_dropout.fit(X_peq, y_peq, validation_data=(X_val, y_val), epochs=25, batch_size=64, verbose=0)

acc_tr_sem = h_sem.history["accuracy"][-1] * 100
acc_va_sem = h_sem.history["val_accuracy"][-1] * 100
acc_tr_com = h_com.history["accuracy"][-1] * 100
acc_va_com = h_com.history["val_accuracy"][-1] * 100

print(f"SEM Dropout -> Treino: {acc_tr_sem:.2f}% | Validação: {acc_va_sem:.2f}% | Gap (Overfitting): {acc_tr_sem - acc_va_sem:.2f}% | Val Loss: {h_sem.history['val_loss'][-1]:.4f}")
print(f"COM Dropout -> Treino: {acc_tr_com:.2f}% | Validação: {acc_va_com:.2f}% | Gap (Overfitting): {acc_tr_com - acc_va_com:.2f}% | Val Loss: {h_com.history['val_loss'][-1]:.4f}")
```

**Análise:** Sem `Dropout`, o modelo crava `100,00%` no treino pequeno (decorou tudo!), mas sofre um *gap* maior e uma perda `val_loss` mais alta (erros mais arrogantes). O `Dropout(0.5)` reduz o *gap* de sobreajuste, contém a explosão da perda na validação e melhora a generalização!

</details>

---

### Exercício 2: Projetando um Filtro Convolucional Detector de Diagonais (`3 × 3`) em `NumPy` Puro
Na Caixa Aberta 2 criamos filtros para bordas horizontais e verticais. Agora projete uma matriz `filtro_diagonal_45` de tamanho `3 × 3` que realce **hastes diagonais inclinadas (`/` — da esquerda-inferior para a direita-superior)**, muito comuns na escrita do número **`"7"`**!
- Dica: coloque valores positivos (`+2.0`) ao longo da diagonal secundária (posições `(0, 2)`, `(1, 1)` e `(2, 0)`) e valores negativos (`-1.0`) nas demais posições, de modo que a soma dos 9 pesos seja `0.0`.
- Aplique `relu(convolucao_2d_manual(X_train[idx_sete_treino], filtro_diagonal_45))` sobre um dígito `"7"` e plote a imagem original ao lado do mapa filtrado.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado do Exercício 2</b></summary>

```python
# Localiza um dígito '7' no conjunto normalizado de treino
idx_sete_treino = int(np.where(y_train == 7)[0][0])
img_sete = X_train[idx_sete_treino]

# Filtro 3x3 sintonizado em traços inclinados a ~45° (diagonal secundária '/')
# A soma total dos elementos vale 3*(+2) + 6*(-1) = 0 (fundo uniforme gera resposta zero!)
filtro_diagonal_45 = np.array(
    [[-1.0, -1.0,  2.0],
     [-1.0,  2.0, -1.0],
     [ 2.0, -1.0, -1.0]], dtype=np.float32
)

mapa_diag = relu(convolucao_2d_manual(img_sete, filtro_diagonal_45))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3.3))
ax1.imshow(img_sete, cmap="gray")
ax1.set_title("Dígito '7' Original (28x28)")
ax1.axis("off")

ax2.imshow(mapa_diag, cmap="magma")
ax2.set_title("Resposta ao Filtro Diagonal '/' (26x26)")
ax2.axis("off")

plt.tight_layout()
plt.show()
```

**Análise:** Repare como a haste inclinada do `"7"` brilha intensamente no mapa da direita, enquanto a barra horizontal superior quase desaparece!

</details>

---

### Exercício 3: Controle de Qualidade Industrial com Limiar de Confiança (`Softmax`)
Em um sistema automatizado real (como leitura óptica de lotes farmacêuticos ou compensação de cheques), **não somos obrigados a aceitar cegamente chutes inseguros**: se a maior probabilidade do vetor `Softmax` ficar abaixo de um limiar de segurança (por exemplo, `85%`), a imagem é desviada para **inspeção humana**!

Usando a matriz `probs_teste_cnn` (formato `(10000, 10)`) e `y_pred_cnn` já calculados nas `10.000` imagens de teste:
1. Calcule a confiança máxima de cada previsão (`confiancas = np.max(probs_teste_cnn, axis=1)`).
2. Crie uma máscara booleana `aprovadas = confiancas >= 0.85`.
3. Imprima: (a) qual porcentagem das `10.000` imagens é resolvida de forma 100% automática, (b) quantas imagens vão para revisão humana, e (c) qual passa a ser a **acurácia da máquina nas imagens aprovadas automaticamente**!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado do Exercício 3</b></summary>

```python
# Maior probabilidade dentre as 10 classes para cada uma das 10.000 imagens de teste
confiancas = np.max(probs_teste_cnn, axis=1)

# Regra de segurança operacional: só aprova automaticamente se P_max >= 85%
limiar = 0.85
aprovadas = confiancas >= limiar

taxa_automacao = np.mean(aprovadas) * 100.0
qtd_revisao_humana = int(np.sum(~aprovadas))
acuracia_aprovadas = accuracy_score(y_test[aprovadas], y_pred_cnn[aprovadas]) * 100.0

print(f"Limiar de confiança adotado               : {limiar * 100:.0f}%")
print(f"Lotes processados automaticamente         : {taxa_automacao:.2f}% ({int(np.sum(aprovadas))} de 10.000)")
print(f"Encaminhados para conferência de operador : {qtd_revisao_humana} imagens ({100 - taxa_automacao:.2f}%)")
print(f"Acurácia nas leituras automáticas         : {acuracia_aprovadas:.2f}%!")
```

**Análise de Engenharia:** Encaminhando apenas uma fração mínima (~1% a 3%) de caligrafias duvidosas para um operador humano, a confiabilidade das leituras automatizadas salta para **acima de 99,6%**, combinando velocidade computacional com segurança industrial!

</details>

---

## 11. Conclusão e Síntese Geral do Problema

Neste caderno construímos, do zero matemático (`NumPy`) à implementação industrial (`TensorFlow/Keras`), toda a cadeia de solução do problema clássico de **Reconhecimento de Dígitos Manuscritos (`MNIST`)**:

### 1. Comparativo Direto das Arquiteturas (`MLP` vs. `CNN`)

| Critério Técnico | `MLP` (Rede Totalmente Conectada) | `CNN` (Rede Neural Convolucional) |
| :--- | :--- | :--- |
| **Representação da Entrada** | Vetor 1D achatado (`Flatten`: `784` números isolados) | Matriz/Tensor espacial 2D (`28 × 28 × 1`) |
| **Noção de Vizinhança Espacial** | **Perdida** — ignora quais pixels estão lado a lado ou acima/abaixo | **Preservada** — varre janelas locais $3 \times 3$ capturando bordas, cantos e curvas |
| **Robustez a Deslocamentos** | **Baixa** — pequenos desvios de posição derrubam a acurácia | **Alta** — compartilhamento de pesos + `MaxPooling` + *Data Augmentation* |
| **Uso Ideal na Engenharia** | Dados tabulares de sensores discretos (pressão, vazão, ensaios químicos) | Imagens, visão industrial, espectrogramas de vibração e mapas térmicos |

### 2. Revisão do Fluxo Completo de 5 Etapas
1. **Entrada Normalizada:** Conversão da matriz de intensidades de 8 bits (`0` a `255`) para escala contínua adimensional `[0, 1]`, respeitando a separação prévia entre **Treino (`50k`)**, **Validação (`10k`)** e **Teste (`10k`)**.
2. **Propagação Direta (*Forward Propagation*):** Extração progressiva de características com ativações não lineares **`ReLU`** nas camadas internas e conversão em distribuição de probabilidades via **`Softmax`** na camada final.
3. **Função de Perda (*Cross-Entropy Loss*):** Medição logarítmica $-\ln(\hat{y}_{\text{real}})$ da divergência entre o vetor previsto e o dígito verdadeiro.
4. **Retropropagação (*Backpropagation*):** Cálculo eficiente dos gradientes ($\nabla \mathbf{W}$) de trás para frente pela Regra da Cadeia, atualizando filtros e pesos via **Gradiente Descendente adaptativo (`Adam`)**.
5. **Saída e Tomada de Decisão:** Vetor de `10` probabilidades que somam `100%`, onde a maior probabilidade define o dígito reconhecido (podendo acionar revisão humana quando a confiança é baixa).

### 3. Soluções para os Desafios Reais
- **Contra a Variação na Caligrafia:** Extração hierárquica de traços por filtros convolucionais combinada ao **Aumento de Dados (*Data Augmentation*)** com pequenas rotações, zoom e translações aleatórias.
- **Contra o *Overfitting* (Sobreajuste):** Monitoramento em conjunto de validação separado, desligamento estocástico de neurônios no treino (**`Dropout`**), aumento sintético da variabilidade de treino e **Parada Antecipada (*Early Stopping*)**.

---

## Próximos Passos no Repositório

- Quer aplicar redes neurais multiclasse e regressão diretamente em **processos químicos, ensaios mecânicos de impressão 3D, petróleo e séries temporais**? Avance para os **[Projetos Aplicados de Engenharia (`engenharia/README.md`)](engenharia/README.md)**!
- Quer revisar qualquer conceito matemático ou de programação utilizado aqui? Consulte o **[Glossário Completo (`GLOSSARIO.md`)](../GLOSSARIO.md)**.